In [1]:
import math
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import torch

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.Alpha9.market.events import EventQueue, MarketEvent, OrderEvent, SignalEvent
from src.Alpha9.pipeline.data_handler import DataHandler
from src.Alpha9.strategy.model import Model
from src.Alpha9.strategy.position.portfolio import Portfolio
from src.Alpha9.strategy.predict import Strategy
from src.Alpha9.strategy.risk.risk_manager import RiskManager
from src.Alpha9.utility import get_config, get_path, read_file

In [4]:
config = get_config.load()
data = read_file.read_data("train", "data")

In [5]:
SYMBOLS = [symbol.split("/")[0] for symbol in config["pipeline"]["symbols"]]
date_format = "%Y-%m-%d %H:%M:%S"
START_DATE = datetime.strptime("2022-01-01 00:00:00", date_format)
END_DATE = datetime.strptime("2022-01-01 23:59:59", date_format)

capital = config["backtest"]["capital"]
transaction_cost_fraction = config["market"]["transaction_cost_fraction"]
bankruptcy_fraction = config["strategy"]["bankruptcy_fraction"]
slippage_cost_fraction = config["strategy"]["slippage_cost_fraction"]
stop_loss_multiple = config["strategy"]["stop_loss_multiple"]
stop_loss_portion = config["strategy"]["stop_loss_portion"]
take_profit_multiple = config["strategy"]["take_profit_multiple"]
take_profit_portion = config["strategy"]["take_profit_portion"]

seq_length = config["strategy"]["sequence_length"]
seq_length = 2
model_dir = get_path.absolute(config["path"]["strategy"]["model"])
portfolio_dir = get_path.absolute(config["path"]["strategy"]["portfolio"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [6]:
event_q = EventQueue()
data_handler = DataHandler(data, SYMBOLS, event_q, START_DATE, END_DATE)
model = Model(SYMBOLS, model_dir)
strategy = Strategy(data_handler, event_q, model, SYMBOLS, seq_length, device)
risk_m = RiskManager(
    data_handler,
    SYMBOLS,
    slippage_cost_fraction,
    stop_loss_multiple,
    stop_loss_portion,
    take_profit_multiple,
    take_profit_portion,
)
portf = Portfolio(
    SYMBOLS, data_handler, event_q, risk_m, capital, bankruptcy_fraction, portfolio_dir
)

In [7]:
events_processed = 0
fiducia = []
orders = []

In [8]:
while data_handler.continue_backtest:
    data_handler.update_candles()
    while not event_q.empty():
        event = event_q.get_event()
        events_processed += 1
        if isinstance(event, MarketEvent):
            portf.update_timeindex(event)
            strategy.calculate_fiducia(event)
        elif isinstance(event, SignalEvent):
            fiducia.append(portf._sanitize(event))
            portf.update_signal(event)
        elif isinstance(event, OrderEvent):
            orders.append(event.description)

In [9]:
portf.all_portfolios

[{'timestamp': Timestamp('2022-01-01 00:00:00'),
  'cash': 1000000.0,
  'total-equity': 0.0,
  'total-transaction-cost': 0.0,
  'portfolio': {'BNB': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}},
   'BTC': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}},
   'ETH': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}},
   'SOL': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}}}},
 {'timestamp': Timestamp('2022-01-01 01:00:00'),
  'cash': 1000000.0,
  'total-equity': 0.0,
  'total-transaction-cost': 0.0,
  'portfolio': {'BNB': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}},
   'BTC': {'amount': 0,
    'stop-loss': {'price': 0, 'portion': 0},
    'take-profit': {'price': 0, 'portion': 0}},
   'ETH': {'amount

In [10]:
events_processed

70

In [11]:
fiducia

[{'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.36552928931500245,
  'BTC': 0.36552928931500245,
  'ETH': 0.36552928931500245,
  'SOL': -0.36552928931500245},
 {'BNB': -0.3655

In [12]:
fiducia[0]["BNB"]

-0.36552928931500245

In [13]:
orders

[{'BNB': {'timestamp': Timestamp('2022-01-01 01:00:00'),
   'amount': np.float64(-706.746499062263),
   'price': np.float64(516.6828),
   'stop-loss': {'price': np.float64(526.1591463764439),
    'portion': np.float64(-565.3971992498103)},
   'take-profit': {'price': np.float64(502.4682804353342),
    'portion': np.float64(-565.3971992498103)}},
  'BTC': {'timestamp': Timestamp('2022-01-01 01:00:00'),
   'amount': np.float64(7.8141048215042845),
   'price': np.float64(46824.918139999994),
   'stop-loss': {'price': np.float64(45835.77187022179),
    'portion': np.float64(6.251283857203428)},
   'take-profit': {'price': np.float64(48308.637544667305),
    'portion': np.float64(6.251283857203428)}},
  'ETH': {'timestamp': Timestamp('2022-01-01 01:00:00'),
   'amount': np.float64(98.13156611738935),
   'price': np.float64(3728.6148899999994),
   'stop-loss': {'price': np.float64(3646.8691883952924),
    'portion': np.float64(78.50525289391149)},
   'take-profit': {'price': np.float64(3851.

In [14]:
orders[0]["BNB"]

{'timestamp': Timestamp('2022-01-01 01:00:00'),
 'amount': np.float64(-706.746499062263),
 'price': np.float64(516.6828),
 'stop-loss': {'price': np.float64(526.1591463764439),
  'portion': np.float64(-565.3971992498103)},
 'take-profit': {'price': np.float64(502.4682804353342),
  'portion': np.float64(-565.3971992498103)}}

In [15]:
price = data_handler.get_latest_candles_value(("close", "BNB"), 100).iloc[1] * 0.999

In [16]:
invested = fiducia[0]["BNB"] * 1000000

In [17]:
print(invested)
print(price)

-365529.28931500245
516.6828


In [18]:
amt = invested / price

In [19]:
amt

np.float64(-707.4539530152782)